<a href="https://colab.research.google.com/github/DivyaMeenaSundaram/Prompt-Engineering/blob/main/Prompt_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PROMPT EVALUATION PRACTICAL
# STAGE 1 - IMPORT LIBRARIES
# ============================================================

# Import pandas.
# Pandas is used to read the Excel dataset and work with tabular data.
import pandas as pd

# Import os.
# os allows us to work with environment variables,
# which we will use later to store the Gemini API key securely.
import os

# Print a message to confirm that the libraries were imported.
print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
GOOGLE_API_KEY = "your-api-key"

In [3]:
# ============================================================
# STAGE 2 - LOAD THE DATASET
# ============================================================

# Specify the name of the Excel file.
# Make sure the Excel file is available in the same folder
# as this notebook, or provide the complete file path.
file_path = "prompt_evaluation_dataset_100_records.xlsx"

# Read the "Dataset" worksheet from the Excel file.
# The result is stored as a pandas DataFrame.
df = pd.read_excel(
    file_path,
    sheet_name="Dataset"
)

# Display the first five records.
# This allows us to verify that the dataset was loaded correctly.
print("First five records:")
print(df.head())

First five records:
     ID                               Question  \
0  R001                  Can I return an item?   
1  R002  How long do I have to return an item?   
2  R003              Can I return a used item?   
3  R004      Do I need a receipt for a return?   
4  R005    Can I return an item after 45 days?   

                                             Context       Category  \
0              Items can be returned within 30 days.  Return policy   
1              Items can be returned within 30 days.  Return policy   
2  Used items may be returned if they meet the re...  Return policy   
3                A receipt is required for a return.  Return policy   
4             The standard return period is 30 days.  Return policy   

                                     Expected_Answer  Split  
0                     Return an item within 30 days.  Train  
1                                           30 days.  Train  
2  Used items can be returned if they meet the re...  Train  
3   

In [4]:
# ============================================================
# STAGE 3 - INSPECT THE DATASET
# ============================================================

# Display the total number of records.
print("\nTotal number of records:")
print(len(df))

# Display the names of all columns.
print("\nDataset columns:")
print(df.columns.tolist())

# Display the number of records assigned to each split.
print("\nTrain/Test distribution:")
print(df["Split"].value_counts())

# Display the number of questions belonging to each category.
print("\nQuestion categories:")
print(df["Category"].value_counts())


Total number of records:
100

Dataset columns:
['ID', 'Question', 'Context', 'Category', 'Expected_Answer', 'Split']

Train/Test distribution:
Split
Train    80
Test     20
Name: count, dtype: int64

Question categories:
Category
Delivery           14
Return policy      13
Account            12
Discount           11
Payment            10
Support            10
Invoice             8
Order tracking      8
Cancellation        5
Exchange policy     3
Refund policy       3
Privacy             2
Security            1
Name: count, dtype: int64


In [5]:
# ============================================================
# STAGE 4 - CREATE TRAINING AND TEST DATASETS
# ============================================================

# Select the records whose Split value is "Train".
# These 80 records are our development/reference dataset.
train_df = df[
    df["Split"] == "Train"
].copy()

# Select the records whose Split value is "Test".
# These 20 records will be used for prompt evaluation.
test_df = df[
    df["Split"] == "Test"
].copy()

# Reset the row numbers of the training dataset.
# This makes the index start from zero again.
train_df = train_df.reset_index(drop=True)

# Reset the row numbers of the test dataset.
test_df = test_df.reset_index(drop=True)

# Display the number of records in each dataset.
print("Development records:", len(train_df))
print("Test records:", len(test_df))

Development records: 80
Test records: 20


In [6]:
# ============================================================
# STAGE 5 - SELECT FIVE TEST QUESTIONS
# ============================================================

# Select five questions from the held-out test dataset.
# These five questions will be used for the live API experiment.
evaluation_df = test_df.head(5).copy()

# Reset the row numbers after selecting the five questions.
evaluation_df = evaluation_df.reset_index(drop=True)

# Display the five questions selected for evaluation.
print("Questions selected for prompt evaluation:\n")

# Loop through the five selected records.
for index, row in evaluation_df.iterrows():

    # Display the question number and question text.
    print(f"{index + 1}. {row['Question']}")

Questions selected for prompt evaluation:

1. What should I do if tracking is not updating?
2. Can I return an item with its packaging removed?
3. Are all damaged products returnable?
4. Can I use UPI for my order?
5. Can I use a coupon during checkout?


In [7]:
# ============================================================
# STAGE 6 - INSPECT THE EVALUATION DATA
# ============================================================

# Select only the important columns for our experiment.
evaluation_view = evaluation_df[
    [
        "ID",
        "Question",
        "Context",
        "Category",
        "Expected_Answer"
    ]
]

# Display the selected records.
print(
    evaluation_view.to_string(index=False)
)

  ID                                         Question                                                                   Context       Category                                        Expected_Answer
R081    What should I do if tracking is not updating?             Check the order tracking page for the latest delivery status. Order tracking                         Check the order tracking page.
R082 Can I return an item with its packaging removed?                     The available information does not provide an answer.  Return policy State that the information does not provide an answer.
R083             Are all damaged products returnable? Damaged items are eligible for return if they meet the return conditions.  Return policy               Only if they meet the return conditions.
R084                      Can I use UPI for my order?                                        UPI is an accepted payment method.        Payment                                  Yes, UPI is accepted.
R085      

In [8]:
# ============================================================
# STAGE 7 - VALIDATE THE DATASET
# ============================================================

# Check whether any of the five questions contain missing values.
missing_questions = evaluation_df["Question"].isna().sum()

# Check whether any contexts are missing.
missing_contexts = evaluation_df["Context"].isna().sum()

# Check whether any expected answers are missing.
missing_expected_answers = evaluation_df[
    "Expected_Answer"
].isna().sum()

# Display the validation results.
print("Missing questions:", missing_questions)
print("Missing contexts:", missing_contexts)
print("Missing expected answers:", missing_expected_answers)

# Check whether everything required for evaluation is available.
if (
    missing_questions == 0
    and missing_contexts == 0
    and missing_expected_answers == 0
):

    # Display a success message.
    print("\nDataset validation successful.")

else:

    # Display a warning if something is missing.
    print("\nWarning: Some required values are missing.")

Missing questions: 0
Missing contexts: 0
Missing expected answers: 0

Dataset validation successful.


In [9]:
# ============================================================
# STAGE 8 - CONNECT TO GEMINI
# ============================================================

# Install the LangChain integration for Google Gemini.
# Run this only if the package is not already installed.
!pip install -q -U langchain-google-genai

# Import the Gemini chat model from LangChain.
from langchain_google_genai import ChatGoogleGenerativeAI

# Import os so that the API key can be read from an environment variable.
import os

# Display a message confirming that the package is ready.
print("Gemini package is ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
Gemini package is ready.


In [ ]:
# ============================================================
# ENTER THE GEMINI API KEY
# ============================================================

# Import getpass.
# getpass allows the API key to be entered without displaying it
# openly on the screen.
from getpass import getpass

# Ask the user to enter the Gemini API key.
# The key will not be visibly displayed while typing.
GOOGLE_API_KEY = getpass("Enter your Gemini API key: ")

# Store the API key in an environment variable.
# The Gemini library can use this environment variable for authentication.
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Display only a confirmation message.
# We deliberately do NOT print the actual API key.
print("API key has been stored securely for this notebook session.")

In [11]:
# ============================================================
# CREATE THE GEMINI MODEL
# ============================================================

# Create the Gemini chat model.
# We do not specify temperature because the selected model
# uses fixed sampling defaults.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

# Display a confirmation message.
# Creating the model object does NOT make an API call.
print("Gemini model created successfully.")

Gemini model created successfully.


In [12]:
# ============================================================
# STAGE 9 - CREATE PROMPT V1
# ============================================================

# Create Prompt V1.
# This is intentionally simple because it will act as our baseline.
prompt_v1 = """
Answer the customer's question.

Use the context provided below.

Question:
{question}

Context:
{context}
"""

# Display Prompt V1.
print("PROMPT V1")
print("-" * 50)
print(prompt_v1)

PROMPT V1
--------------------------------------------------

Answer the customer's question.

Use the context provided below.

Question:
{question}

Context:
{context}



In [13]:
# ============================================================
# STAGE 10 - CREATE PROMPT V2
# ============================================================

# Create Prompt V2.
# This prompt contains explicit instructions about grounding,
# relevance, and handling unknown information.
prompt_v2 = """
You are a customer-support assistant.

Answer the customer's question using ONLY the information
provided in the context.

Follow these instructions:
1. Give a direct answer to the customer's question.
2. Use only information supported by the context.
3. Do not invent facts, policies, prices, dates, or conditions.
4. Do not add information that is unrelated to the question.
5. If the context does not contain enough information to answer,
   say:
   "The available information does not provide an answer."
6. Keep the answer concise and specific.

Question:
{question}

Context:
{context}
"""

# Display Prompt V2.
print("PROMPT V2")
print("-" * 50)
print(prompt_v2)

PROMPT V2
--------------------------------------------------

You are a customer-support assistant.

Answer the customer's question using ONLY the information
provided in the context.

Follow these instructions:
1. Give a direct answer to the customer's question.
2. Use only information supported by the context.
3. Do not invent facts, policies, prices, dates, or conditions.
4. Do not add information that is unrelated to the question.
5. If the context does not contain enough information to answer,
   say:
   "The available information does not provide an answer."
6. Keep the answer concise and specific.

Question:
{question}

Context:
{context}



In [14]:
# ============================================================
# STAGE 11 - CREATE A REUSABLE GENERATION FUNCTION
# ============================================================

# Define a function for sending one question to Gemini.
#
# The function receives:
#   prompt   -> the prompt version we want to use
#   question -> the customer's question
#   context  -> the information available to the model
def generate_answer(prompt, question, context):

    # Insert the question and context into the prompt template.
    formatted_prompt = prompt.format(
        question=question,
        context=context
    )

    # Send the completed prompt to Gemini.
    # THIS LINE MAKES ONE API CALL.
    response = llm.invoke(formatted_prompt)

    # Extract the text generated by Gemini.
    answer = response.content

    # Return the generated answer.
    return answer

In [16]:
# ============================================================
# STAGE 12 - TEST THE GEMINI CONNECTION
# ============================================================

# Select the first evaluation question.
test_question = evaluation_df.loc[0, "Question"]

# Select the context belonging to that question.
test_context = evaluation_df.loc[0, "Context"]

# Generate an answer using Prompt V1.
# This makes ONE Gemini API call.
test_answer = generate_answer(
    prompt_v1,
    test_question,
    test_context
)

# Display the question.
print("QUESTION:")
print(test_question)

# Display the context.
print("\nCONTEXT:")
print(test_context)

# Display Gemini's generated answer.
print("\nGENERATED ANSWER:")
print(test_answer)

QUESTION:
What should I do if tracking is not updating?

CONTEXT:
Check the order tracking page for the latest delivery status.

GENERATED ANSWER:
[{'type': 'text', 'text': 'Based on the provided information, you should check the order tracking page for the latest delivery status.', 'extras': {'signature': 'Et4ICtsIARFNMg+AD3ryc6JOtKEriP6pTPXl5Q+qIu8t9DdNBokMH1l6A3KbTlQGpkIp/JqdlXlS45oTTl7WehtD/1Gd9mNSSBr+1ulWjHLfv8N0Tquc1cU2Qq+7Qy6RPhjFwENwr3idpDWa5yEcB9o9c1Cvg++CUE8rdybS+vvP5+1fnfX/ROeKV8DRG8wIyzIVKtOKwjz1C2T5MoSqFk5beQr9/UhiDZEdInFoo9IRZ4hiXA7eBpbTEB5a4I83qqOKrJKb3+bjyZ8ZxpWJLcKDrCLZsH+n5H8Qbjx2iOYXJf4uXcBwzL4Ig3eJPTYwPu0FPPRnUNzdq84ljO02ghVyeJCWTRA6Jwl0KxtknQRuysoCCdDGwx1/9kD73gVo9v0EVwKjMaQcoERp1sJmB7UacvC36xonBMz2qDjZOtBT8LiP4QWHs8VZQz117SACCsid/T81rXLjNe+aoN6T61zF44duLtKx62dl1CinwAF5n7FLBVWemFwpyH2Ck63N8ipSC/2q1V/jxJPatA/nxWpZ31FV0/TnO0qS2VHS2bCNRaS5ZonMcwwsq1ZvIY45HFqlrz4GUxTAXLrw0LbOxg2LJMLUcLc2iiiSkSMxoNjf9nQnELPhi0piwGqbbNcqKNfApThzSJT+hKTXhmHLkxDjaEP85tT5nSxIFyJMhIHrI28pLLg

In [17]:
# ============================================================
# STAGE 13 - GENERATE PROMPT V1 RESPONSES
# ============================================================

# Create an empty list to store all V1 answers.
v1_answers = []

# Add the answer from our Stage 12 test.
# This avoids making another API call for the same question.
v1_answers.append(test_answer)

# Start from the second evaluation question.
# The first question was already processed in Stage 12.
for index in range(1, len(evaluation_df)):

    # Get the current question.
    question = evaluation_df.loc[index, "Question"]

    # Get the context for the current question.
    context = evaluation_df.loc[index, "Context"]

    # Generate the answer using Prompt V1.
    # Each iteration makes ONE API call.
    answer = generate_answer(
        prompt_v1,
        question,
        context
    )

    # Store the generated answer.
    v1_answers.append(answer)

    # Display progress so we know which question is being processed.
    print(f"V1 response generated for question {index + 1}/5.")

# Add the V1 answers to our evaluation DataFrame.
evaluation_df["V1_Answer"] = v1_answers

V1 response generated for question 2/5.
V1 response generated for question 3/5.
V1 response generated for question 4/5.
V1 response generated for question 5/5.


In [18]:
# ============================================================
# STAGE 14 - INSPECT PROMPT V1 RESULTS
# ============================================================

# Display the question, expected answer, and V1 generated answer.
print(
    evaluation_df[
        [
            "ID",
            "Question",
            "Expected_Answer",
            "V1_Answer"
        ]
    ].to_string(index=False)
)

  ID                                         Question                                        Expected_Answer                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [19]:
# ============================================================
# PROMPT EVALUATION PRACTICAL
# STAGE 15 - GENERATE PROMPT V2 ANSWERS
# ============================================================

# Create an empty list to store the answers generated by Prompt V2.
v2_answers = []

# Loop through all 5 questions selected for evaluation.
for index in range(len(evaluation_df)):

    # Get the current question from the evaluation dataset.
    question = evaluation_df.loc[index, "Question"]

    # Get the context associated with the current question.
    context = evaluation_df.loc[index, "Context"]

    # Send the question and context to Gemini using Prompt V2.
    answer = generate_answer(
        prompt_v2,
        question,
        context
    )

    # Store the generated answer in the list.
    v2_answers.append(answer)

    # Display progress so students can see each API call being completed.
    print(f"V2 response generated for question {index + 1}/5.")

# Add all V2 answers as a new column in the evaluation dataframe.
evaluation_df["V2_Answer"] = v2_answers

V2 response generated for question 1/5.
V2 response generated for question 2/5.
V2 response generated for question 3/5.
V2 response generated for question 4/5.
V2 response generated for question 5/5.


In [20]:
# ============================================================
# STAGE 16 - DISPLAY V1 AND V2 ANSWERS
# ============================================================

# Loop through every evaluation record.
for index, row in evaluation_df.iterrows():

    # Print a separator to make each question easier to read.
    print("\n" + "=" * 80)

    # Display the question number.
    print(f"QUESTION {index + 1}")

    # Display the original customer question.
    print("\nQuestion:")
    print(row["Question"])

    # Display the context given to the model.
    print("\nContext:")
    print(row["Context"])

    # Display the expected answer from the dataset.
    print("\nExpected Answer:")
    print(row["Expected_Answer"])

    # Display the answer produced by Prompt V1.
    print("\nPrompt V1 Answer:")
    print(row["V1_Answer"])

    # Display the answer produced by Prompt V2.
    print("\nPrompt V2 Answer:")
    print(row["V2_Answer"])


QUESTION 1

Question:
What should I do if tracking is not updating?

Context:
Check the order tracking page for the latest delivery status.

Expected Answer:
Check the order tracking page.

Prompt V1 Answer:
[{'type': 'text', 'text': 'Based on the provided information, you should check the order tracking page for the latest delivery status.', 'extras': {'signature': 'Et4ICtsIARFNMg+AD3ryc6JOtKEriP6pTPXl5Q+qIu8t9DdNBokMH1l6A3KbTlQGpkIp/JqdlXlS45oTTl7WehtD/1Gd9mNSSBr+1ulWjHLfv8N0Tquc1cU2Qq+7Qy6RPhjFwENwr3idpDWa5yEcB9o9c1Cvg++CUE8rdybS+vvP5+1fnfX/ROeKV8DRG8wIyzIVKtOKwjz1C2T5MoSqFk5beQr9/UhiDZEdInFoo9IRZ4hiXA7eBpbTEB5a4I83qqOKrJKb3+bjyZ8ZxpWJLcKDrCLZsH+n5H8Qbjx2iOYXJf4uXcBwzL4Ig3eJPTYwPu0FPPRnUNzdq84ljO02ghVyeJCWTRA6Jwl0KxtknQRuysoCCdDGwx1/9kD73gVo9v0EVwKjMaQcoERp1sJmB7UacvC36xonBMz2qDjZOtBT8LiP4QWHs8VZQz117SACCsid/T81rXLjNe+aoN6T61zF44duLtKx62dl1CinwAF5n7FLBVWemFwpyH2Ck63N8ipSC/2q1V/jxJPatA/nxWpZ31FV0/TnO0qS2VHS2bCNRaS5ZonMcwwsq1ZvIY45HFqlrz4GUxTAXLrw0LbOxg2LJMLUcLc2iiiSkSMxoNjf9nQnELPhi

In [21]:
# ============================================================
# STAGE 17 - SAVE GENERATED ANSWERS
# ============================================================

# Define the name of the output Excel file.
output_file = "prompt_evaluation_generated_responses.xlsx"

# Save the evaluation dataframe to an Excel file.
evaluation_df.to_excel(
    output_file,
    index=False
)

# Confirm that the file has been saved.
print(f"Generated responses saved to: {output_file}")

Generated responses saved to: prompt_evaluation_generated_responses.xlsx


In [22]:
# ============================================================
# STAGE 18A - CREATE THE AUTOMATED EVALUATION PROMPT
# ============================================================

# Create an empty list to hold all response records.
evaluation_records = []

# Loop through the five questions in the evaluation dataset.
for index, row in evaluation_df.iterrows():

    # Create a record for the Prompt V1 response.
    v1_record = f"""
RECORD_ID: {row["ID"]}_V1
PROMPT_VERSION: V1

QUESTION:
{row["Question"]}

CONTEXT:
{row["Context"]}

EXPECTED ANSWER:
{row["Expected_Answer"]}

GENERATED ANSWER:
{row["V1_Answer"]}
"""

    # Add the V1 record to the evaluation list.
    evaluation_records.append(v1_record)

    # Create a record for the Prompt V2 response.
    v2_record = f"""
RECORD_ID: {row["ID"]}_V2
PROMPT_VERSION: V2

QUESTION:
{row["Question"]}

CONTEXT:
{row["Context"]}

EXPECTED ANSWER:
{row["Expected_Answer"]}

GENERATED ANSWER:
{row["V2_Answer"]}
"""

    # Add the V2 record to the evaluation list.
    evaluation_records.append(v2_record)

# Combine all ten records into one large evaluation input.
all_evaluation_records = "\n".join(evaluation_records)

# Display how many responses will be evaluated.
print("Responses prepared for evaluation:", len(evaluation_records))

Responses prepared for evaluation: 10


In [23]:
# ============================================================
# STAGE 18B - DEFINE EVALUATION CRITERIA
# ============================================================

# Create the instructions that Gemini will use as an evaluator.
evaluation_prompt = f"""
You are an evaluator for a prompt engineering experiment.

Evaluate each generated customer-support answer using ONLY the
question, context, expected answer, and generated answer provided.

Evaluate these four metrics:

1. ACCURACY
   Score 1 if the generated answer correctly answers the question.
   Score 0 if the answer is incorrect, materially incomplete,
   or contradicts the expected answer or context.

2. RELEVANCE
   Give a score from 1 to 5.
   5 = directly answers the question with no unnecessary content.
   4 = mostly direct with minor unnecessary content.
   3 = partially direct or contains noticeable extra content.
   2 = largely off-topic.
   1 = does not address the question.

3. HALLUCINATION
   Score 1 if the answer contains information that is not supported
   by the supplied context.
   Score 0 if every factual claim is supported by the context.

4. SPECIFICITY
   Give a score from 1 to 5.
   5 = precise and contains the concrete information needed.
   4 = sufficiently specific with minor vagueness.
   3 = moderately specific.
   2 = vague or missing important details.
   1 = extremely vague or non-informative.

IMPORTANT:
- Do not judge based on writing style alone.
- Do not require exact wording matching.
- A correct answer may use different wording from the expected answer.
- Distinguish between an incorrect answer and a hallucinated answer.
- For questions where the context does not provide enough information,
  correctly stating that the information is unavailable should receive
  a high accuracy score.
- Base every judgment on the supplied information.

Return ONLY valid JSON.

Use this exact structure:

[
  {{
    "record_id": "ID_V1",
    "prompt_version": "V1",
    "accuracy": 0,
    "relevance": 1,
    "hallucination": 0,
    "specificity": 1,
    "reason": "Brief explanation"
  }}
]

Evaluate all records.

RESPONSES TO EVALUATE:
{all_evaluation_records}
"""

# Display the first part of the evaluator prompt for verification.
print(evaluation_prompt[:2000])
print("\nEvaluation prompt prepared successfully.")


You are an evaluator for a prompt engineering experiment.

Evaluate each generated customer-support answer using ONLY the
question, context, expected answer, and generated answer provided.

Evaluate these four metrics:

1. ACCURACY
   Score 1 if the generated answer correctly answers the question.
   Score 0 if the answer is incorrect, materially incomplete,
   or contradicts the expected answer or context.

2. RELEVANCE
   Give a score from 1 to 5.
   5 = directly answers the question with no unnecessary content.
   4 = mostly direct with minor unnecessary content.
   3 = partially direct or contains noticeable extra content.
   2 = largely off-topic.
   1 = does not address the question.

3. HALLUCINATION
   Score 1 if the answer contains information that is not supported
   by the supplied context.
   Score 0 if every factual claim is supported by the context.

4. SPECIFICITY
   Give a score from 1 to 5.
   5 = precise and contains the concrete information needed.
   4 = sufficient

In [24]:
# ============================================================
# STAGE 18C - RUN AUTOMATED EVALUATION
# ============================================================

# Send the complete evaluation prompt to Gemini.
evaluation_response = llm.invoke(evaluation_prompt)

# Extract the text returned by Gemini.
evaluation_text = evaluation_response.content

# Display the raw evaluator response.
print("RAW EVALUATION RESPONSE:")
print(evaluation_text)

RAW EVALUATION RESPONSE:
[{'type': 'text', 'text': '```json\n[\n  {\n    "record_id": "R081_V1",\n    "prompt_version": "V1",\n    "accuracy": 1,\n    "relevance": 5,\n    "hallucination": 0,\n    "specificity": 5,\n    "reason": "The generated answer accurately, directly, and specifically answers the question using the provided context without hallucination."\n  },\n  {\n    "record_id": "R081_V2",\n    "prompt_version": "V2",\n    "accuracy": 0,\n    "relevance": 2,\n    "hallucination": 0,\n    "specificity": 1,\n    "reason": "The generated answer incorrectly claims the information is unavailable, even though the context explicitly provides instructions."\n  },\n  {\n    "record_id": "R082_V1",\n    "prompt_version": "V1",\n    "accuracy": 1,\n    "relevance": 5,\n    "hallucination": 0,\n    "specificity": 5,\n    "reason": "The answer correctly states that the information is unavailable, directly aligning with the context and expected response."\n  },\n  {\n    "record_id": "R082

In [29]:
# ============================================================
# STAGE 18D - PARSE THE EVALUATION RESULTS
# ============================================================

# Import Python's built-in JSON library.
import json

# Store the content returned by Gemini.
evaluation_content = evaluation_response.content

# Initialize evaluation_results
evaluation_results = []

# Check whether Gemini has already returned a Python list.
if isinstance(evaluation_content, list):
    # If it's a list, we expect the JSON string to be in the 'text' key of the first item
    if evaluation_content and 'text' in evaluation_content[0]:
        clean_evaluation_text = evaluation_content[0]['text']
        # Remove a possible Markdown JSON code-fence.
        clean_evaluation_text = clean_evaluation_text.replace("```json", "")
        # Remove the closing Markdown code-fence.
        clean_evaluation_text = clean_evaluation_text.replace("```", "")
        # Remove unnecessary spaces and line breaks at the beginning/end.
        clean_evaluation_text = clean_evaluation_text.strip()
        # Convert the JSON text into a Python list of dictionaries.
        evaluation_results = json.loads(clean_evaluation_text)
    else:
        raise ValueError("Expected evaluation_content list to contain a dictionary with a 'text' key.")
# Otherwise, check whether Gemini returned the result as a raw JSON string.
elif isinstance(evaluation_content, str):
    # Store the text returned by Gemini.
    clean_evaluation_text = evaluation_content

    # Remove a possible Markdown JSON code-fence.
    clean_evaluation_text = clean_evaluation_text.replace("```json", "")

    # Remove the closing Markdown code-fence.
    clean_evaluation_text = clean_evaluation_text.replace("```", "")

    # Remove unnecessary spaces and line breaks at the beginning/end.
    clean_evaluation_text = clean_evaluation_text.strip()

    # Convert the JSON text into a Python list of dictionaries.
    evaluation_results = json.loads(clean_evaluation_text)

# Handle any unexpected response format.
else:

    # Stop execution and explain the problem.
    raise TypeError(
        "Unexpected response format: "
        + str(type(evaluation_content))
    )

# Convert the evaluation results into a pandas DataFrame.
metrics_df = pd.DataFrame(evaluation_results)

# Display the structured evaluation results.
print("EVALUATION RESULTS")
print("-" * 80)

# Print the DataFrame without the default row numbers.
print(metrics_df.to_string(index=False))

# Confirm the number of responses evaluated.
print("\nNumber of responses evaluated:", len(metrics_df))

EVALUATION RESULTS
--------------------------------------------------------------------------------
record_id prompt_version  accuracy  relevance  hallucination  specificity                                                                                                                             reason
  R081_V1             V1         1          5              0            5 The generated answer accurately, directly, and specifically answers the question using the provided context without hallucination.
  R081_V2             V2         0          2              0            1  The generated answer incorrectly claims the information is unavailable, even though the context explicitly provides instructions.
  R082_V1             V1         1          5              0            5         The answer correctly states that the information is unavailable, directly aligning with the context and expected response.
  R082_V2             V2         1          5              0            5       

In [30]:
# ============================================================
# STAGE 18E - CALCULATE OVERALL METRICS
# ============================================================

# Group the evaluation results according to prompt version.
summary_df = (
    metrics_df
    .groupby("prompt_version")
    .agg(
        Accuracy=("accuracy", "mean"),
        Relevance=("relevance", "mean"),
        Hallucination_Rate=("hallucination", "mean"),
        Specificity=("specificity", "mean")
    )
    .reset_index()
)

# Convert accuracy from a proportion into a percentage.
summary_df["Accuracy"] = summary_df["Accuracy"] * 100

# Convert hallucination rate from a proportion into a percentage.
summary_df["Hallucination_Rate"] = summary_df["Hallucination_Rate"] * 100

# Display the final metric summary.
print("PROMPT PERFORMANCE SUMMARY")
print("-" * 70)
print(summary_df.to_string(index=False))

PROMPT PERFORMANCE SUMMARY
----------------------------------------------------------------------
prompt_version  Accuracy  Relevance  Hallucination_Rate  Specificity
            V1     100.0        5.0                 0.0          5.0
            V2      80.0        4.4                 0.0          4.2


In [32]:
# ============================================================
# STAGE 18F - SAVE EVALUATION RESULTS
# ============================================================

# Create an Excel writer so multiple sheets can be saved together.
with pd.ExcelWriter(
    "prompt_evaluation_results.xlsx",
    engine="openpyxl"
) as writer:

    # Save the original questions and generated responses.
    evaluation_df.to_excel(
        writer,
        sheet_name="Generated_Answers",
        index=False
    )

    # Save the individual evaluation scores.
    metrics_df.to_excel(
        writer,
        sheet_name="Metric_Scores",
        index=False
    )

    # Save the aggregated V1 versus V2 results.
    summary_df.to_excel(
        writer,
        sheet_name="Prompt_Summary",
        index=False
    )

# Confirm successful saving.
print("Evaluation results saved successfully.")

Evaluation results saved successfully.


In [33]:
# ============================================================
# STAGE 19A - SELECT QUESTIONS FOR CONSISTENCY TESTING
# ============================================================

# Select the first two evaluation questions for the consistency test.
consistency_df = evaluation_df.head(2).copy()

# Reset the dataframe index for easier access.
consistency_df = consistency_df.reset_index(drop=True)

# Display the questions selected for consistency testing.
print("Questions selected for consistency testing:\n")

# Loop through the selected questions.
for index, row in consistency_df.iterrows():

    # Display the question number and question text.
    print(f"{index + 1}. {row['Question']}")

Questions selected for consistency testing:

1. What should I do if tracking is not updating?
2. Can I return an item with its packaging removed?


In [34]:
# ============================================================
# STAGE 19B - REPEAT PROMPT V1 FOR CONSISTENCY
# ============================================================

# Create an empty list to store the repeated V1 answers.
v1_repeat_answers = []

# Loop through the two selected consistency questions.
for index, row in consistency_df.iterrows():

    # Retrieve the question from the dataset.
    question = row["Question"]

    # Retrieve the original context.
    context = row["Context"]

    # Run the SAME Prompt V1 again.
    repeat_answer = generate_answer(
        prompt_v1,
        question,
        context
    )

    # Store the repeated answer.
    v1_repeat_answers.append(repeat_answer)

    # Display the completed repetition.
    print(f"Repeated V1 response generated for question {index + 1}/2.")

# Store the repeated V1 answers in the consistency dataframe.
consistency_df["V1_Repeat_Answer"] = v1_repeat_answers

Repeated V1 response generated for question 1/2.
Repeated V1 response generated for question 2/2.


In [35]:
# ============================================================
# STAGE 19C - REPEAT PROMPT V2 FOR CONSISTENCY
# ============================================================

# Create an empty list to store the repeated V2 answers.
v2_repeat_answers = []

# Loop through the two selected consistency questions.
for index, row in consistency_df.iterrows():

    # Retrieve the question from the dataset.
    question = row["Question"]

    # Retrieve the original context.
    context = row["Context"]

    # Run the SAME Prompt V2 again.
    repeat_answer = generate_answer(
        prompt_v2,
        question,
        context
    )

    # Store the repeated answer.
    v2_repeat_answers.append(repeat_answer)

    # Display the completed repetition.
    print(f"Repeated V2 response generated for question {index + 1}/2.")

# Store the repeated V2 answers in the consistency dataframe.
consistency_df["V2_Repeat_Answer"] = v2_repeat_answers

Repeated V2 response generated for question 1/2.
Repeated V2 response generated for question 2/2.


In [36]:
# ============================================================
# STAGE 19D - DISPLAY CONSISTENCY RESULTS
# ============================================================

# Loop through the two consistency records.
for index, row in consistency_df.iterrows():

    # Print a separator for readability.
    print("\n" + "=" * 80)

    # Display the question number.
    print(f"QUESTION {index + 1}")

    # Display the question.
    print("\nQuestion:")
    print(row["Question"])

    # Display the original V1 answer.
    print("\nOriginal V1 Answer:")
    print(row["V1_Answer"])

    # Display the repeated V1 answer.
    print("\nRepeated V1 Answer:")
    print(row["V1_Repeat_Answer"])

    # Display the original V2 answer.
    print("\nOriginal V2 Answer:")
    print(row["V2_Answer"])

    # Display the repeated V2 answer.
    print("\nRepeated V2 Answer:")
    print(row["V2_Repeat_Answer"])


QUESTION 1

Question:
What should I do if tracking is not updating?

Original V1 Answer:
[{'type': 'text', 'text': 'Based on the provided information, you should check the order tracking page for the latest delivery status.', 'extras': {'signature': 'Et4ICtsIARFNMg+AD3ryc6JOtKEriP6pTPXl5Q+qIu8t9DdNBokMH1l6A3KbTlQGpkIp/JqdlXlS45oTTl7WehtD/1Gd9mNSSBr+1ulWjHLfv8N0Tquc1cU2Qq+7Qy6RPhjFwENwr3idpDWa5yEcB9o9c1Cvg++CUE8rdybS+vvP5+1fnfX/ROeKV8DRG8wIyzIVKtOKwjz1C2T5MoSqFk5beQr9/UhiDZEdInFoo9IRZ4hiXA7eBpbTEB5a4I83qqOKrJKb3+bjyZ8ZxpWJLcKDrCLZsH+n5H8Qbjx2iOYXJf4uXcBwzL4Ig3eJPTYwPu0FPPRnUNzdq84ljO02ghVyeJCWTRA6Jwl0KxtknQRuysoCCdDGwx1/9kD73gVo9v0EVwKjMaQcoERp1sJmB7UacvC36xonBMz2qDjZOtBT8LiP4QWHs8VZQz117SACCsid/T81rXLjNe+aoN6T61zF44duLtKx62dl1CinwAF5n7FLBVWemFwpyH2Ck63N8ipSC/2q1V/jxJPatA/nxWpZ31FV0/TnO0qS2VHS2bCNRaS5ZonMcwwsq1ZvIY45HFqlrz4GUxTAXLrw0LbOxg2LJMLUcLc2iiiSkSMxoNjf9nQnELPhi0piwGqbbNcqKNfApThzSJT+hKTXhmHLkxDjaEP85tT5nSxIFyJMhIHrI28pLLg0MRh+5nFTZAQ3BXDGhQvEnEZlvkJPtJujgCKdV2qhFI+8hUgtChPHS1Bw

In [42]:
# ============================================================
# STAGE 19E - CALCULATE CONSISTENCY SCORE
# ============================================================

# Import SequenceMatcher from Python's standard library.
from difflib import SequenceMatcher

# Define a function to calculate textual consistency.
def calculate_consistency(original_answer, repeated_answer):

    # Extract text from the list of dictionaries and convert to lowercase.
    # Assuming the content is always in the 'text' key of the first item in the list.
    original_text = original_answer[0]['text'].lower() if isinstance(original_answer, list) and original_answer else ""
    repeated_text = repeated_answer[0]['text'].lower() if isinstance(repeated_answer, list) and repeated_answer else ""

    # Calculate the similarity ratio between the two responses.
    score = SequenceMatcher(
        None,
        original_text,
        repeated_text
    ).ratio()

    # Return the calculated score.
    return score

# Calculate V1 consistency for each selected question.
consistency_df["V1_Consistency"] = consistency_df.apply(
    lambda row: calculate_consistency(
        row["V1_Answer"],
        row["V1_Repeat_Answer"]
    ),
    axis=1
)

# Calculate V2 consistency for each selected question.
consistency_df["V2_Consistency"] = consistency_df.apply(
    lambda row: calculate_consistency(
        row["V2_Answer"],
        row["V2_Repeat_Answer"]
    ),
    axis=1
)

# Convert the scores into percentages.
consistency_df["V1_Consistency_Percent"] = (
    consistency_df["V1_Consistency"] * 100
)

consistency_df["V2_Consistency_Percent"] = (
    consistency_df["V2_Consistency"] * 100
)

# Display the consistency scores.
print(
    consistency_df[
        [
            "ID",
            "V1_Consistency_Percent",
            "V2_Consistency_Percent"
        ]
    ].to_string(index=False)
)


  ID  V1_Consistency_Percent  V2_Consistency_Percent
R081               93.333333                   100.0
R082               76.691729                   100.0


In [40]:
# ============================================================
# STAGE 19E - CALCULATE CONSISTENCY SCORE
# ============================================================

# Import SequenceMatcher from Python's standard library.
from difflib import SequenceMatcher

# Define a function to convert a model response into plain text.
def response_to_text(response):

    # Check whether the response is already a string.
    if isinstance(response, str):

        # Return the string directly.
        return response

    # Check whether the response is a list.
    elif isinstance(response, list):

        # Create an empty list to store extracted text parts.
        text_parts = []

        # Examine every item inside the response list.
        for item in response:

            # Check whether the item is a dictionary.
            if isinstance(item, dict):

                # Extract the text value if a "text" field exists.
                if "text" in item:
                    text_parts.append(str(item["text"]))

                # Otherwise, convert the dictionary to text.
                else:
                    text_parts.append(str(item))

            # If the item is not a dictionary, convert it to text.
            else:
                text_parts.append(str(item))

        # Combine all extracted parts into one string.
        return " ".join(text_parts)

    # Handle any other response type safely.
    else:

        # Convert the response to a string.
        return str(response)


# Define a function to calculate textual consistency.
def calculate_consistency(original_answer, repeated_answer):

    # Convert the original response into plain text.
    original_text = response_to_text(original_answer)

    # Convert the repeated response into plain text.
    repeated_text = response_to_text(repeated_answer)

    # Convert both responses to lowercase.
    original_text = original_text.lower()
    repeated_text = repeated_text.lower()

    # Calculate the similarity ratio between the two responses.
    score = SequenceMatcher(
        None,
        original_text,
        repeated_text
    ).ratio()

    # Return the consistency score.
    return score


# Calculate V1 consistency for each selected question.
consistency_df["V1_Consistency"] = consistency_df.apply(
    lambda row: calculate_consistency(
        row["V1_Answer"],
        row["V1_Repeat_Answer"]
    ),
    axis=1
)


# Calculate V2 consistency for each selected question.
consistency_df["V2_Consistency"] = consistency_df.apply(
    lambda row: calculate_consistency(
        row["V2_Answer"],
        row["V2_Repeat_Answer"]
    ),
    axis=1
)


# Convert V1 consistency scores into percentages.
consistency_df["V1_Consistency_Percent"] = (
    consistency_df["V1_Consistency"] * 100
)


# Convert V2 consistency scores into percentages.
consistency_df["V2_Consistency_Percent"] = (
    consistency_df["V2_Consistency"] * 100
)


# Display the consistency scores.
print(
    consistency_df[
        [
            "ID",
            "V1_Consistency_Percent",
            "V2_Consistency_Percent"
        ]
    ].to_string(index=False)
)

  ID  V1_Consistency_Percent  V2_Consistency_Percent
R081               93.333333                   100.0
R082               76.691729                   100.0


In [44]:
# ============================================================
# STAGE 19F - CALCULATE AVERAGE CONSISTENCY
# ============================================================

# Calculate the average consistency percentage for Prompt V1.
v1_consistency = (
    consistency_df["V1_Consistency_Percent"].mean()
)

# Calculate the average consistency percentage for Prompt V2.
v2_consistency = (
    consistency_df["V2_Consistency_Percent"].mean()
)

# Display the final consistency results.
print("CONSISTENCY RESULTS")
print("-" * 50)

# Display V1 consistency.
print(f"Prompt V1 Consistency: {v1_consistency:.2f}%")

# Display V2 consistency.
print(f"Prompt V2 Consistency: {v2_consistency:.2f}%")

CONSISTENCY RESULTS
--------------------------------------------------
Prompt V1 Consistency: 85.01%
Prompt V2 Consistency: 100.00%


In [45]:
# ============================================================
# STAGE 19G - ADD CONSISTENCY TO SUMMARY
# ============================================================

# Calculate the average consistency for V1.
v1_consistency = consistency_df["V1_Consistency_Percent"].mean()

# Calculate the average consistency for V2.
v2_consistency = consistency_df["V2_Consistency_Percent"].mean()

# Create a dictionary containing the average consistency scores.
consistency_summary = {
    "V1": v1_consistency,
    "V2": v2_consistency
}

# Add a new Consistency column to the summary dataframe.
summary_df["Consistency"] = summary_df["prompt_version"].map(
    consistency_summary
)

# Display the updated summary.
print("UPDATED PROMPT PERFORMANCE SUMMARY")
print("-" * 80)

print(
    summary_df[
        [
            "prompt_version",
            "Accuracy",
            "Relevance",
            "Hallucination_Rate",
            "Specificity",
            "Consistency"
        ]
    ].to_string(index=False)
)


UPDATED PROMPT PERFORMANCE SUMMARY
--------------------------------------------------------------------------------
prompt_version  Accuracy  Relevance  Hallucination_Rate  Specificity  Consistency
            V1     100.0        5.0                 0.0          5.0    85.012531
            V2      80.0        4.4                 0.0          4.2   100.000000


In [46]:
# ============================================================
# STAGE 19H - SAVE CONSISTENCY RESULTS
# ============================================================

# Create an Excel writer for the final intermediate results.
with pd.ExcelWriter(
    "prompt_evaluation_results_with_consistency.xlsx",
    engine="openpyxl"
) as writer:

    # Save the generated V1 and V2 responses.
    evaluation_df.to_excel(
        writer,
        sheet_name="Generated_Answers",
        index=False
    )

    # Save the individual metric scores.
    metrics_df.to_excel(
        writer,
        sheet_name="Metric_Scores",
        index=False
    )

    # Save the consistency experiment.
    consistency_df.to_excel(
        writer,
        sheet_name="Consistency",
        index=False
    )

    # Save the overall prompt comparison.
    summary_df.to_excel(
        writer,
        sheet_name="Prompt_Summary",
        index=False
    )

# Confirm that the file was created.
print("Results saved successfully.")

Results saved successfully.


In [47]:
# ============================================================
# STAGE 20A - PREPARE A/B TEST DATA
# ============================================================

# Create a dictionary containing the consistency results.
consistency_values = {
    "V1": v1_consistency,
    "V2": v2_consistency
}

# Create a copy of the overall metric summary.
ab_summary = summary_df.copy()

# Add the consistency values to the summary.
ab_summary["Consistency"] = (
    ab_summary["prompt_version"].map(consistency_values)
)

# Display the metrics that will be used for A/B testing.
print("METRICS USED FOR A/B TESTING")
print("-" * 70)

print(
    ab_summary[
        [
            "prompt_version",
            "Accuracy",
            "Relevance",
            "Hallucination_Rate",
            "Specificity",
            "Consistency"
        ]
    ].to_string(index=False)
)

METRICS USED FOR A/B TESTING
----------------------------------------------------------------------
prompt_version  Accuracy  Relevance  Hallucination_Rate  Specificity  Consistency
            V1     100.0        5.0                 0.0          5.0    85.012531
            V2      80.0        4.4                 0.0          4.2   100.000000


In [48]:
# ============================================================
# STAGE 20B - CREATE A/B TEST PROMPT
# ============================================================

# Create a compact representation of the measured metrics.
metric_comparison = ab_summary[
    [
        "prompt_version",
        "Accuracy",
        "Relevance",
        "Hallucination_Rate",
        "Specificity",
        "Consistency"
    ]
].to_string(index=False)

# Create a list to store the representative V1/V2 responses.
ab_response_records = []

# Loop through all five evaluation questions.
for index, row in evaluation_df.iterrows():

    # Add the question and both generated answers.
    record = f"""
QUESTION {index + 1}:
{row["Question"]}

EXPECTED ANSWER:
{row["Expected_Answer"]}

PROMPT V1 ANSWER:
{row["V1_Answer"]}

PROMPT V2 ANSWER:
{row["V2_Answer"]}
"""

    # Store the comparison record.
    ab_response_records.append(record)

# Combine all response records into one text block.
ab_responses = "\n".join(ab_response_records)

# Create the final A/B evaluation prompt.
ab_prompt = f"""
You are evaluating two prompts in a prompt engineering experiment.

The two prompts are:

PROMPT V1:
{prompt_v1}

PROMPT V2:
{prompt_v2}

The measured evaluation metrics are:

{metric_comparison}

The generated answers are:

{ab_responses}

Perform an A/B comparison of Prompt V1 and Prompt V2.

Consider the following criteria:

1. Accuracy
2. Relevance
3. Hallucination rate
4. Specificity
5. Consistency

Important:
- Higher accuracy is better.
- Higher relevance is better.
- Lower hallucination rate is better.
- Higher specificity is better.
- Higher consistency is better.
- Do not choose a prompt merely because its answer is longer.
- Prefer answers that are correct, relevant, specific, and supported by
  the supplied context.
- Use the measured metrics as the primary evidence.
- Use the individual answers to explain the observed differences.

Return ONLY valid JSON using exactly this structure:

{{
    "winner": "V1 or V2",
    "decision": "Brief explanation of why the selected prompt performs better.",
    "accuracy_comparison": "Brief comparison",
    "relevance_comparison": "Brief comparison",
    "hallucination_comparison": "Brief comparison",
    "specificity_comparison": "Brief comparison",
    "consistency_comparison": "Brief comparison",
    "recommendation": "Which prompt should be used and why."
}}
"""

# Confirm that the A/B test prompt has been created.
print("A/B evaluation prompt prepared successfully.")

A/B evaluation prompt prepared successfully.


In [49]:
# ============================================================
# STAGE 20C - RUN FINAL A/B TEST
# ============================================================

# Send the A/B evaluation prompt to Gemini.
ab_response = llm.invoke(ab_prompt)

# Extract the text returned by Gemini.
ab_result_text = ab_response.content

# Display the raw A/B evaluation result.
print("RAW A/B TEST RESULT:")
print(ab_result_text)

RAW A/B TEST RESULT:
[{'type': 'text', 'text': '```json\n{\n    "winner": "V1",\n    "decision": "Prompt V1 significantly outperforms Prompt V2 across key performance metrics, achieving perfect scores in Accuracy (100.0 vs. 80.0), Relevance (5.0 vs. 4.4), and Specificity (5.0 vs. 4.2), while keeping a 0.0% Hallucination Rate. Prompt V2\'s overly rigid refusal instruction caused it to falsely claim that available context did not contain enough information (e.g., Question 1), leading to lower accuracy and helpfulness.",\n    "accuracy_comparison": "Prompt V1 achieved 100.0% accuracy by correctly answering all questions based on context, whereas Prompt V2 achieved only 80.0% due to incorrectly issuing fallback refusals for answerable questions.",\n    "relevance_comparison": "Prompt V1 scored a perfect 5.0 in relevance compared to Prompt V2\'s 4.4, as V1 provided relevant, actionable answers while V2 occasionally produced non-responsive fallback statements.",\n    "hallucination_compariso

In [51]:
# ============================================================
# STAGE 20D - PARSE A/B TEST RESULT
# ============================================================

# Extract the string content from the list returned by Gemini.
# This handles cases where the response is a list of dictionaries.
if isinstance(ab_result_text, list) and ab_result_text and 'text' in ab_result_text[0]:
    clean_ab_text = ab_result_text[0]['text']
else:
    # If it's not a list, assume it's already a string or handle other unexpected types.
    clean_ab_text = str(ab_result_text)

# Remove possible Markdown JSON code fences.
clean_ab_text = clean_ab_text.replace("```json", "")

# Remove any remaining closing code fence.
clean_ab_text = clean_ab_text.replace("```", "")

# Remove unnecessary whitespace.
clean_ab_text = clean_ab_text.strip()

# Convert the JSON response into a Python dictionary.
ab_result = json.loads(clean_ab_text)

# Display the selected winner.
print("A/B TEST WINNER:")
print(ab_result["winner"])

# Display the overall decision.
print("\nDECISION:")
print(ab_result["decision"])

# Display the recommendation.
print("\nRECOMMENDATION:")
print(ab_result["recommendation"])


A/B TEST WINNER:
V1

DECISION:
Prompt V1 significantly outperforms Prompt V2 across key performance metrics, achieving perfect scores in Accuracy (100.0 vs. 80.0), Relevance (5.0 vs. 4.4), and Specificity (5.0 vs. 4.2), while keeping a 0.0% Hallucination Rate. Prompt V2's overly rigid refusal instruction caused it to falsely claim that available context did not contain enough information (e.g., Question 1), leading to lower accuracy and helpfulness.

RECOMMENDATION:
Use Prompt V1 because it delivers fully accurate, relevant, and specific answers without hallucinating. To further refine V1, minor formatting constraints can be introduced to increase consistency without sacrificing its ability to answer correctly.


In [52]:
# ============================================================
# STAGE 20E - DISPLAY COMPLETE A/B ANALYSIS
# ============================================================

# Display the comparison for accuracy.
print("\nACCURACY COMPARISON:")
print(ab_result["accuracy_comparison"])

# Display the comparison for relevance.
print("\nRELEVANCE COMPARISON:")
print(ab_result["relevance_comparison"])

# Display the comparison for hallucination.
print("\nHALLUCINATION COMPARISON:")
print(ab_result["hallucination_comparison"])

# Display the comparison for specificity.
print("\nSPECIFICITY COMPARISON:")
print(ab_result["specificity_comparison"])

# Display the comparison for consistency.
print("\nCONSISTENCY COMPARISON:")
print(ab_result["consistency_comparison"])

# Display the final recommendation.
print("\nFINAL RECOMMENDATION:")
print(ab_result["recommendation"])


ACCURACY COMPARISON:
Prompt V1 achieved 100.0% accuracy by correctly answering all questions based on context, whereas Prompt V2 achieved only 80.0% due to incorrectly issuing fallback refusals for answerable questions.

RELEVANCE COMPARISON:
Prompt V1 scored a perfect 5.0 in relevance compared to Prompt V2's 4.4, as V1 provided relevant, actionable answers while V2 occasionally produced non-responsive fallback statements.

HALLUCINATION COMPARISON:
Both V1 and V2 achieved a 0.0% hallucination rate, successfully ensuring that no invented facts or ungrounded details were generated.

SPECIFICITY COMPARISON:
Prompt V1 achieved higher specificity (5.0 vs. 4.2) because it gave direct, detailed answers matching expected criteria, while V2's non-answer fallback lowered its overall specificity.

CONSISTENCY COMPARISON:
Prompt V2 demonstrated higher consistency (100.0 vs. 85.01), driven by its strict negative constraints and template-based fallback phrasing.

FINAL RECOMMENDATION:
Use Prompt V

In [53]:
# ============================================================
# STAGE 20F - SAVE COMPLETE PROMPT EVALUATION
# ============================================================

# Define the name of the final Excel output file.
final_output_file = "final_prompt_evaluation_results.xlsx"

# Create an Excel writer for the final workbook.
with pd.ExcelWriter(
    final_output_file,
    engine="openpyxl"
) as writer:

    # Save the five questions used for evaluation and their answers.
    evaluation_df.to_excel(
        writer,
        sheet_name="Generated_Answers",
        index=False
    )

    # Save the individual metric scores.
    metrics_df.to_excel(
        writer,
        sheet_name="Metric_Scores",
        index=False
    )

    # Save the consistency experiment.
    consistency_df.to_excel(
        writer,
        sheet_name="Consistency",
        index=False
    )

    # Save the overall V1 versus V2 metrics.
    ab_summary.to_excel(
        writer,
        sheet_name="Prompt_Summary",
        index=False
    )

    # Convert the A/B result dictionary into a dataframe.
    ab_result_df = pd.DataFrame(
        [ab_result]
    )

    # Save the final A/B decision.
    ab_result_df.to_excel(
        writer,
        sheet_name="AB_Test",
        index=False
    )

# Confirm that the complete workbook was created.
print(f"Final results saved to: {final_output_file}")

Final results saved to: final_prompt_evaluation_results.xlsx


In [54]:
# ============================================================
# STAGE 21 - FINAL EXPERIMENTAL COMPARISON
# ============================================================

# Create a copy of the summary for final analysis.
final_comparison = ab_summary.copy()

# Display the complete metric table.
print("FINAL PROMPT COMPARISON")
print("=" * 80)

print(
    final_comparison[
        [
            "prompt_version",
            "Accuracy",
            "Relevance",
            "Hallucination_Rate",
            "Specificity",
            "Consistency"
        ]
    ].to_string(index=False)
)

# Display the LLM-based A/B winner.
print("\n" + "=" * 80)
print("LLM-BASED A/B WINNER:", ab_result["winner"])

# Display the final recommendation.
print("\nFINAL RECOMMENDATION:")
print(ab_result["recommendation"])

FINAL PROMPT COMPARISON
prompt_version  Accuracy  Relevance  Hallucination_Rate  Specificity  Consistency
            V1     100.0        5.0                 0.0          5.0    85.012531
            V2      80.0        4.4                 0.0          4.2   100.000000

LLM-BASED A/B WINNER: V1

FINAL RECOMMENDATION:
Use Prompt V1 because it delivers fully accurate, relevant, and specific answers without hallucinating. To further refine V1, minor formatting constraints can be introduced to increase consistency without sacrificing its ability to answer correctly.
